In [1]:
#라이브러리 불러오기 및 세팅
import requests
import json
import pandas as pd
from bs4 import BeautifulSoup
import numpy as np
import datetime as dt
import xlrd
import datetime
import math

pd.set_option("display.max_columns", None)

#아이디
# ["UNI00415", "UNI00414", "UNI00118"]
#비밀번호
PWD = "skku00221!"
# "qp72017900!!"

In [2]:
#세팅2
import glob
glob.glob("*")

['desktop.ini',
 'MOE-Copy1.ipynb',
 'MOE.ipynb',
 'MOE_Final-ing.ipynb',
 'whole_gls_20210114.xlsx',
 '코로나대응학생정보등록조회(신입생).xlsx',
 '코로나대응학생정보등록조회.xlsx']

# 1. 중요파일 불러오기 및 초기 셋팅

In [3]:
daynow = dt.datetime.today().strftime("%Y%m%d")
daynow = dt.datetime.strptime(daynow,"%Y%m%d")
today = dt.datetime.today().strftime("%Y%m%d")
daynow

datetime.datetime(2021, 1, 14, 0, 0)

In [4]:
#엑셀 데이터 읽어오기,파일 이름 날짜에 주의
#df_skku = pd.read_excel('코로나대응학생정보등록조회.xlsx', sheet_name='sheet')

In [5]:
#맨 앞행 처리
#df_skku = df_skku[df_skku.번호 > 0]

In [6]:
#행 이름 다름
##df_skku.rename(columns = {"기숙사 거주 여부": "기숙사거주여부"}, inplace = True)

In [7]:
#재학생 데이터 (행,열)
##df_skku.shape

In [8]:
#코드의 공백을 없애기
#df_skku.loc[:,'코드'] = df_skku.코드.map(lambda x: x.replace(" ",""))
#df_skku.loc[:,'코드'] = df_skku.코드.map(lambda x: x.replace("-",""))

In [9]:
#df_skku.columns

In [10]:
#df_skku.shape

In [11]:
#df_skku

In [12]:
#신입생과 재학생을 합친 테이블
#df_skku.to_excel("whole_gls_" + today + ".xlsx",index=False)

In [13]:
#df1 = pd.read_excel('코로나대응학생정보등록조회.xlsx', sheet_name='sheet')
#df2 = pd.read_excel('코로나대응학생정보등록조회(신입생).xlsx', sheet_name='sheet')


In [14]:
####### 혹시 신입생과 재학생 파일이 따로 존재하는 시기일 때 (전에 쓰던 코드) ##########
#엑셀 데이터 읽어오기,파일 이름 날짜에 주의, 현재는 두개의 테이블이므로 합쳐야함
df1 = pd.read_excel('코로나대응학생정보등록조회.xlsx', sheet_name='sheet')
df2 = pd.read_excel('코로나대응학생정보등록조회(신입생).xlsx', sheet_name='sheet')
#맨 앞행 처리
df_old = df1[df1.번호 > 0]
df_fresh = df2[df2.번호 > 0]
#코드의 공백을 없애기
df_old['코드'] = df_old['코드'].map(lambda x: x.replace(" ",""))
df_old['코드'] = df_old['코드'].map(lambda x: x.replace("-",""))
df_fresh['코드'] = df_fresh['코드'].map(lambda x: x.replace(" ",""))
df_fresh['코드'] = df_fresh['코드'].map(lambda x: x.replace("-",""))



In [15]:
#df_fresh_new = df_fresh.iloc[:,0:28].copy()
#df_fresh_new2 = df_fresh.iloc[:,28:].copy()


In [16]:
#df_fresh

In [17]:
df_fresh.insert(28,"등록여부",df_fresh[0],True)
df_fresh.insert(29,"2020-2학기신입생여부",df_fresh[0],True)

In [18]:
#df_fresh

In [19]:
#df_old

In [20]:
#재학생과 신입생 데이터 합치기


#'코드'를 키로 해서 outer join
df_skku = df_fresh.merge(df_old, how='outer', on='코드')

df_skku['코드'].value_counts().sort_values()

df_skku.loc[:,'임시'] = df_skku.생년월일_y.map(lambda x: "신입생" if x != x else "재학생")
df_skku['임시'][~(df_skku['생년월일_x'].isna() | df_skku['생년월일_y'].isna())] = "중복"
#열을 예쁘게 만들자
df_total = df_skku
#신입생(x)과 신입생아님(y)로 데이터프레임을 두개로 나눈다
df_new = df_total[df_total['임시'].isin(['신입생'])]
df_last = df_total[df_total['임시'].isin(['재학생','중복'])]

C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  # Remove the CWD from sys.path while we load stuff.


In [21]:
#df_last.iloc[:,99]

In [22]:

#열을 고른다
df_new2 = df_new.iloc[:,0:99].copy()
df_last2 = df_last.iloc[:,99:197].copy()
df_new3 = df_new2.drop(['코드'],axis=1)

#코드와 임시는 맨뒤에 따로 붙인다
s_code = df_last['코드'].values
s1_code = df_last['임시'].values
df_last2['코드'] = s_code
df_last2['임시'] = s1_code

s2_code = df_new['코드'].values
s3_code = df_new['임시'].values
df_new3['코드'] = s2_code
df_new3['임시'] = s3_code
#concat을 써서 붙인다
#컬럼 이름 정리
a= df_new3.columns
d = []


In [23]:
#a[95:97]

In [24]:
#df_last2

In [25]:
#a[99]

In [26]:
for i in a[:-2]:
   d.append(i[:-2])
d.append(a[98])
d.append(a[99])
df_new3.columns = d
#컬럼 이름 정리
a= df_last2.columns
d = []
for i in a[:-2]:
   d.append(i[:-2])
d.append(a[98])
d.append(a[99])
df_last2.columns = d
df_skku = pd.concat([df_new3,df_last2])
df_skku.rename(columns = {"임시": "신입여부"}, inplace = True)
#신입생과 재학생을 합친 테이블
df_skku.to_excel("whole_gls_" + today + ".xlsx",index=False)

In [27]:
#df_new3

In [28]:
df_skku

,0,번호,생년월일,학번,한글성명,영문성명,여권번호,외국인등록번호,외국인등록번호(FIMS),국적,국적구분,성별,입학구분,학생구분,現전형유형,캠퍼스,통합학부,학위,대학원구분,대학원명,학과전공,입학당시우편번호,입학당시주소,재학여부,학적상태,최종학적변동,최종학적변동일,등록여부,2020-2학기신입생여부,설문조사응답이메일,설문조사응답전화번호,설문조사응답위챗,입국진행-비자발급,입국진행-항공예약,입국진행-격리거소,자가격리(예정)거소,입국(예정)일,입국예정구분,기숙사거주여부,입국후한국내주소(또는기숙사 명칭),기숙사,출발일,입국일,출국일,현상태,입력일,GLS입력여부,첨부서류 구분,처리상태,GLS외부이메일,GLS내부이메일,GLS등록주소,GLS등록연락처,입국경과일,최근14일이내,자가격리,유증상여부,주거형태,교육부제공연락처,연락처(핸드폰)(학생입력),이메일(학생입력),위챗(학생입력),카카오톡(학생입력),우편번호,한국거주지주소1(학생입력),한국거주지주소2(학생입력),증상-1일차,증상-2일차,증상-3일차,증상-4일차,증상-5일차,증상-6일차,증상-7일차,증상-8일차,증상-9일차,증상-10일차,증상-11일차,증상-12일차,증상-13일차,증상-14일차,격리-1일차,격리-2일차,격리-3일차,격리-4일차,격리-5일차,격리-6일차,격리-7일차,격리-8일차,격리-9일차,격리-10일차,격리-11일차,격리-12일차,격리-13일차,격리-14일차,등록자,등록일,수정자,수정일,코드,신입여부
0,0.0,1.0,900502.0,2.021730e+09,김성진,KIM SUNG JIN,NaN,9005025100118,9005025100118,캐나다,중국외,남,일반신입학,외국인,일반학생,인사캠,GSB분야,석사,전문대학원,경영전문대학원,SKK GSB Professional MBA 분야,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sjkim365@gmail.com,sjkim365@g.skku.edu,경기도 안양시 만안구 성결대학로64번길 24안양동 2층,,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,900502KIMSUNGJIN,신입생
1,0.0,2.0,960530.0,2.021730e+09,ELÉONORE VAN MARCKE,ELÉONORE VAN MARCKE,NaN,9605306999999,9605306999999,벨기에,중국외,여,일반신입학,외국인,일반학생,인사캠,GSB분야,석사,전문대학원,경영전문대학원,Master In Management Studies,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,eleonore.vanmarcke@gmail.com,eleonorevm@g.skku.edu,Wilrijk ANTWERPSint Bavo straat 89,+32470337349,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,960530ELÉONOREVANMARCKE,신입생
2,0.0,3.0,971222.0,2.021730e+09,ILAN HOURCHANI,ILAN HOURCHANI,NaN,9712225999999,9712225999999,프랑스,중국외,남,일반신입학,외국인,일반학생,인사캠,GSB분야,석사,전문대학원,경영전문대학원,Master In Management Studies,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ilan.hourchani@edhec.com,ihourchani@g.skku.edu,PARIS FRANCE5 Rue Olivier Noyer,0668152077,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,971222ILANHOURCHANI,신입생
3,0.0,4.0,980810.0,2.021730e+09,THOMAS CONANEC,THOMAS CONANEC,NaN,9808105999999,9808105999999,프랑스,중국외,남,일반신입학,외국인,일반학생,인사캠,GSB분야,석사,전문대학원,경영전문대학원,Master In Management Studies,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,thomas.conanec@edhec.com,thomascnc@g.skku.edu,ACIGNE Bretagne12 Le boulais,0299043408,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,980810THOMASCONANEC,신입생
4,0.0,5.0,980925.0,2.021730e+09,LISON MARIE ROSE URSO,LISON MARIE ROSE URSO,NaN,9809256999999,9809256999999,프랑스,중국외,여,일반신입학,외국인,일반학생,인사캠,GSB분야,석사,전문대학원,경영전문대학원,Master In Management Studies,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,lison.urso@g.skku.edu,,,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,980925LISONMARIEROSEURSO,신입생
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5631,0.0,4913.0,781011.0,2.008712e+09,박문걸,PIAO WENJIE,NaN,7.81012e+12,NaN

In [29]:
#a

# 2. 맞춤형 시스템 정보 가져오기

In [30]:
payload_one = lambda passport_dtl: {'TAB_VAL': '0',
 'ADM_YN': 'N',
 'CHK_PAS': '',
 'CHK_ROW': '',
 'CHK_NUM': '',
 'CHK_UNI': '',
 'PASPORT_DTL': passport_dtl,
 'SCH_NAME': '',
 'SCH_UNIV': '',
 'SCH_PAS': '',
 'SCH_ORG': '',
 'SCH_UPD': '',
 'SCH_FRGNR': '',
 'userDept': '53067000',
 'PASPORT_NO': '',
 'ORGIN_COUNTRY': '',
 'USER_NAME': '',
 'UNIV_NM': '',
 'FRGNR_NO': '',
 'UPDT_YN': '',
 'SUB_PASPORT': '',
 'SUB_DT': 'C'}

def add_payload_col(df):
    for col in ['SCH_NAME', 'SCH_UNIV', 'SCH_FRGNR', 'SCH_PAS', 'SCH_UPD', 'SCH_CLASS']:
        df[col] = None
    return df

update_moe_key = ['STUDABRD_SEQ', 'SCH_FRGNR', 'SCH_CLASS', 'LEAVE_DT', 'SCH_NAME', 'SCH_UNIV', 'SCH_PAS', 'SCH_UPD', 'ERCCR_YN', 'FOND_GB', 'ELSM_GB', 'UNIV_NM', 'CAMPUS', 'ORGIN_COUNTRY', 'COURSE_GB', 'OCCP_NM', 'FRSHMN_YN', 'GRADE', 'USER_NAME', 'CLASS_NO', 'MBTLNUM', 'PASPORT_NO', 'SOJO_QUALF', 'FRGNR_NO', 'BIRTH', 'UNIVFCL_YN', 'CUR_RESDNC', 'ERCCR_DT', 'START_ARPRT', 'ADMINISTZONE', 'PASPORT_ISSU', 'SYMPTMS_YN', 'ATTENDANCE_YN', 'MNTRNG_YN', 'CONTACT_YN', 'SKNRGS_STS']

In [31]:
class BizinfoCrawler:
    def __init__(self, user_id, user_pw):
        self._user_id = user_id
        self._pwd = user_pw
        self.s = requests.Session()
        self.china_df = None
        self.world_df = None
                
    def login(self):
        self.s.get('https://bizinfo.moe.go.kr/')
        print(self.s.cookies['JSESSIONID'])
        headers = {
          'Accept': 'application/json, text/javascript, */*; q=0.01',
          'Accept-Encoding': 'gzip, deflate, br',
          'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7',
          'Connection': 'keep-alive',
          'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
          'Cookie': 'JSESSIONID={}'.format(self.s.cookies['JSESSIONID']),
          'Host': 'bizinfo.moe.go.kr',
          'Origin': 'https://bizinfo.moe.go.kr',
          'Referer': 'https://bizinfo.moe.go.kr/SiaLoginForm.do?',
          'Sec-Fetch-Dest': 'empty',
          'Sec-Fetch-Mode': 'cors',
          'Sec-Fetch-Site': 'same-origin',
          'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_3) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/80.0.3987.132 Safari/537.36',
          'X-Requested-With': 'XMLHttpRequest'
        }
        url = "https://bizinfo.moe.go.kr/uat/uia/SiaLogin.do"
        payload = { "userId" : self._user_id, "userPwd" : self._pwd}
        res = self.s.post(url, headers=headers, data=payload)
        print(res.text)
        
    def get_china_df(self):
        URL = "https://bizinfo.moe.go.kr/cbis/sia/ajaxSelectStudabrdList.do"
        headers = {
          'Host': 'bizinfo.moe.go.kr',
          'Connection': 'keep-alive',
          'Accept': 'application/json, text/javascript, */*; q=0.01',
          'Sec-Fetch-Dest': 'empty',
          'X-Requested-With': 'XMLHttpRequest',
          'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_3) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/80.0.3987.132 Safari/537.36',
          'Content-Type': 'application/x-www-form-urlencoded',
          'Origin': 'https://bizinfo.moe.go.kr',
          'Sec-Fetch-Site': 'same-origin',
          'Sec-Fetch-Mode': 'cors',
          'Referer': 'https://bizinfo.moe.go.kr/cbis/sia/studabrdInfoMain.do?menu_no=6500000001',
          'Accept-Encoding': 'gzip, deflate, br',
          'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7',
          'Cookie': 'JSESSIONID={}'.format(self.s.cookies['JSESSIONID']),
        }

        payload = {'CHK_EXCEL': ' ',
        'ADM_YN': 'N',
        'STUDABRD_SEQ': ' ',
        'userDept': '7008261',
        'SCH_FRGNR': ' ',
        'SCH_CLASS': ' ',
        'SCH_NAME': ' ',
        'SCH_UNIV': ' ',
        'SCH_PAS': ' ',
        'SCH_UPD': ' ',
        'CHK_PAS': ' ',
        'CHK_ROW': ' ',
        'CHK_UNI': ' ',
        'CHK_NUM': ' ',
        'STU_CNT': '3032',
        'NONE_DEP': '376',
        'PASPORT_NO': ' ',
        'UPDT_YN': ' ',
        'FRGNR_NO': ' ',
        'CLASS_NO': ' ',
        'USER_NAME': ' ',
        'UNIV_NM': ' ',
        'curentPageNo': '1',
        'pageUnit': '3000',
        'pageSize': '10'}
        response = requests.request("POST", URL, headers=headers, data = payload)
        resultArr = response.json()['data']
        df_new = pd.json_normalize(resultArr)
        return df_new
    

    def update_moe(self, df):
        UPDATE_URL = "https://bizinfo.moe.go.kr/cbis/sia/updateStudabrdInfo.do"
        headers = {
            "Accept":"application/json, text/javascript, */*; q=0.01",
            "Accept-Encoding":"gzip, deflate, br",
            "Accept-Language":"ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
            "Connection":"keep-alive",
            "Content-Length":"155",
            "Content-Type":"application/x-www-form-urlencoded; charset=UTF-8",
            "Cookie": 'JSESSIONID={}'.format(self.s.cookies['JSESSIONID']),
            "Host":"bizinfo.moe.go.kr",
            "Origin":"https://bizinfo.moe.go.kr",
            "Referer":"https://bizinfo.moe.go.kr/cbis/sia/studabrdInfoMain.do?menu_no=6500000001",
            "Sec-Fetch-Mode":"cors",
            "Sec-Fetch-Site":"same-origin",
            "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/79.0.3945.130 Safari/537.36",
            "X-Requested-With":"XMLHttpRequest"
        }
        df = add_payload_col(df)
        for i in range(df.shape[0]):
            payload_test = df[update_moe_key].iloc[i,:].to_dict()
            res = requests.request("POST", UPDATE_URL, headers=headers, data=payload_test)
            if res.status_code != 200:
                print(res.text)
    
    def get_world_df(self):
        URL = "https://bizinfo.moe.go.kr/cbis/sia/ajaxSelectOccdntList.do"
        headers = {
          'Host': 'bizinfo.moe.go.kr',
          'Connection': 'keep-alive',
          'Accept': 'application/json, text/javascript, */*; q=0.01',
          'Sec-Fetch-Dest': 'empty',
          'X-Requested-With': 'XMLHttpRequest',
          'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_3) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/80.0.3987.132 Safari/537.36',
          'Content-Type': 'application/x-www-form-urlencoded',
          'Origin': 'https://bizinfo.moe.go.kr',
          'Sec-Fetch-Site': 'same-origin',
          'Sec-Fetch-Mode': 'cors',
          'Referer': 'https://bizinfo.moe.go.kr/cbis/sia/studabrdInfoMain.do?menu_no=6500000001',
          'Accept-Encoding': 'gzip, deflate, br',
          'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7',
          'Cookie': 'JSESSIONID={}'.format(self.s.cookies['JSESSIONID']),
        }
        
        payload = {'TAB_VAL': '0',
         'ADM_YN': 'N',
         'CHK_PAS': '',
         'CHK_ROW': '',
         'CHK_NUM': '',
         'CHK_UNI': '',
         'PASPORT_DTL': '',
         'SCH_NAME': '',
         'SCH_UNIV': '',
         'SCH_PAS': '',
         'SCH_ORG': '',
         'SCH_UPD': '',
         'SCH_FRGNR': '',
         'userDept': '53067000',
         'PASPORT_NO': '',
         'ORGIN_COUNTRY': '',
         'USER_NAME': '',
         'UNIV_NM': '',
         'FRGNR_NO': '',
         'UPDT_YN': '',
         'SUB_PASPORT': '',
         'SUB_DT': 'C',
         'curentPageNo': '1',
         'pageUnit': '3000',
         'pageSize': '10'}
        response = requests.request("POST", URL, headers=headers, data = payload)
        resultArr = response.json()['data']
        df = pd.json_normalize(resultArr)
        
        if df.shape[0] == 0:
            return df

        one_url = "https://bizinfo.moe.go.kr/cbis/sia/selectOccdntDtl.do"
        headers = {
              'Host': 'bizinfo.moe.go.kr',
              'Connection': 'keep-alive',
              'Accept': 'application/json, text/javascript, */*; q=0.01',
              'Sec-Fetch-Dest': 'empty',
              'X-Requested-With': 'XMLHttpRequest',
              'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_3) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/80.0.3987.132 Safari/537.36',
              'Content-Type': 'application/x-www-form-urlencoded',
              'Origin': 'https://bizinfo.moe.go.kr',
              'Sec-Fetch-Site': 'same-origin',
              'Sec-Fetch-Mode': 'cors',
              'Referer': 'https://bizinfo.moe.go.kr/cbis/sia/studabrdInfoMain.do?menu_no=6500000001',
              'Accept-Encoding': 'gzip, deflate, br',
              'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7',
              'Cookie': 'JSESSIONID={}'.format(self.s.cookies['JSESSIONID']),
            }
        
        payload_one_arr = [payload_one(psno) for psno in df.PASPORT_NO.values]
        for idx, payload in enumerate(payload_one_arr):
            response = requests.request("POST", one_url, headers=headers, data = payload)
            soup = BeautifulSoup(response.text, 'html.parser')
            frgnr_no = soup.select('#FRGNR_NO')[0]['value']
            leave_dt = soup.select('#LEAVE_DT')[0]['value']
            erccr_dt = soup.select('#ERCCR_DT')[0]['value']
            if frgnr_no:
                df.loc[idx,'FRGNR_NO'] = frgnr_no
                df.loc[idx,'BIRTH'] = frgnr_no[:6]
            df.loc[idx,'LEAVE_DT'] = leave_dt
            df.loc[idx,'ERCCR_DT'] = erccr_dt

        return df
    
        
    def close(self):
        self.s.close()

In [32]:
# 크롤러 객체 생성
agents = [BizinfoCrawler(uid, PWD) for uid in ["UNI00118", "UNI00414", "UNI00415"]]

In [33]:
# 로그인
for agent in agents:
    agent.login()

uB2naWHN9xg8hV8ZrtYS2CUT8iNaUmzf187SvCCwSkIdWWLpJdaiBGvZvzzPu1al.moe-biz-extwas2_servlet_engine1
["N"]
uJHx9gsg12Olr0hdrOvkaQPSvbUbkcDA1C1dvWLhynm06gEyJGcssVxWvzvgZ1Ob.moe-biz-extwas1_servlet_engine1
["N"]
urgD6CoyHW5c9aybrwg8Dj6JNDrAdt8c1nJGLcU5ypRjJYflJLjIoChayGHIuRkN.moe-biz-extwas2_servlet_engine1
["N"]


In [34]:
#데이터 긁어오기,오래 걸림
df_world = pd.concat([agent.get_world_df() for agent in agents])
df_world.reset_index(drop=True,inplace=True)

JSONDecodeError: Expecting value: line 3 column 1 (char 4)

In [ ]:
#날짜 데이터형태로 바꾸기
df_world.ERCCR_DT = df_world.ERCCR_DT.map(lambda x: dt.datetime.strptime(x,'%Y%m%d'))
df_world.LEAVE_DT = df_world.LEAVE_DT.map(lambda x: dt.datetime.strptime(x,'%Y%m%d') if x==x and x else np.nan)

In [ ]:
#교육부 데이터의 (행,열)
df_world.shape

In [ ]:
#교육부 데이터 출력
#df_world

In [ ]:
#크롤링해서 받아온 교육부 데이터 엑셀에 출력해보기
df_update2 = df_world
df_update2.to_excel("교육부 데이터_" + today + ".xlsx",index=False)

# 3. 학교 중요 파일 Merge

In [ ]:
#대전환 데이터
#df_skku

In [ ]:
#교육부 데이터
#df_world

In [ ]:
#병합 함수 정의
def join_students(df_gov, df_skku, find_by_name=True):
    df_gov.loc[:,'코드'] = df_gov.BIRTH.map(lambda x: str(x)[-6:]) + df_gov.USER_NAME.map(lambda x: x if x==x else '')
    #df_gov['코드'] = df_world.BIRTH.map(lambda x: str(x)[-6:]) + df_world.USER_NAME.map(lambda x: x if x==x else '')
    df_gov.loc[:,'코드'] = df_gov.loc[:,'코드'].map(lambda x: x.replace(" ",""))
    df_gov.loc[:,'코드'] = df_gov.loc[:,'코드'].map(lambda x: x.replace("-",""))
    df_result = df_gov.merge(df_skku, how='left', on='코드')
    df_mid = df_gov.merge(df_skku, how='right', on='코드')
    
    #코드로 매칭된 친구와 아닌 친구를 나눈다-교육부
    condition = df_result['번호'] == df_result['번호']
    df_temp = df_result[condition]
    #df_temp.to_excel("임시1_" + today + ".xlsx",index=False)
    df_temp2 = df_result[~condition]
    df_temp2 = df_temp2.loc[:, df_temp2.columns[:25]]
    
    #코드로 매칭된 친구와 아닌 친구를 나눈다-gls
    condition = df_mid['UNIV_NM'] == df_mid['UNIV_NM']
    df_temp3 = df_mid[condition]
    #df_temp3.to_excel("임시3_" + today + ".xlsx",index=False)
    df_temp4 = df_mid[~condition]
    df_temp4 = df_temp4.loc[:, df_temp4.columns[25:]]
    
    df_result2 = df_temp2.merge(df_temp4, how='outer', left_on ='CLASS_NO', right_on = '학번')
    #df_result2.to_excel("임시2_" + today + ".xlsx",index=False)
    df_result3 = pd.concat([df_temp,df_result2])
    #df_result2 = df_result.merge(df_skku, how='outer', left_on ='CLASS_NO', right_on = '학번')
    return df_result3

In [ ]:
df_world.CLASS_NO = df_world.CLASS_NO.map(lambda x: np.str(x))

In [ ]:
df_skku.학번 = df_skku.학번.map(lambda x: np.str(x))

In [ ]:
print(df_world.shape)
print(df_skku.shape)

In [ ]:
#교육부 데이터에서 K1:국내 체류중 K2:출국함 설정
df_world['ERCCR_YN'] = 'K1' 
df_world.loc[df_world.LEAVE_DT > df_world.ERCCR_DT,'ERCCR_YN'] = 'K2'

In [ ]:
#출국한 날짜별로 몇명있는지 보여주기
df_world[df_world.ERCCR_YN == 'K2'].LEAVE_DT.value_counts().sort_index()

In [ ]:
#교육부와 대전환 병합
df_world_join = join_students(df_world, df_skku, find_by_name=False)

In [ ]:
#병합한 데이터 (행,열)
df_world_join.shape

In [ ]:
#df_world_join.to_excel("임시_" + today + ".xlsx",index=False)

In [ ]:
#병합한 데이터 출력
df_world_join.shape

In [ ]:
#df_world_join 

In [ ]:
df_temp = df_world_join.copy()

In [ ]:
#현상태업데이트-빈 값은 입국일도 보고 바꾸기
series = np.where(pd.notnull(df_temp['ERCCR_DT']) == True, df_temp['ERCCR_DT'], df_temp['입국일'])

In [ ]:
df_temp['LEAVE_DT'] = df_temp['LEAVE_DT'].map(lambda x: x if x==x else dt.datetime(1970, 1, 1, 0, 0))
df_temp['현상태'] = np.where(series > df_temp['LEAVE_DT'], "입국", df_temp['현상태'])
df_temp['현상태'] = np.where(series <= df_temp['LEAVE_DT'], "출국", df_temp['현상태'])
df_temp['LEAVE_DT'] = df_temp['LEAVE_DT'].map(lambda x: x if x!=dt.datetime(1970, 1, 1, 0, 0) else np.nan)

In [ ]:
#ERCCR_DT, LEAVE_DT가 있으면 입국일 열이랑 출국일 열을 해당 값으로 덮기
df_temp['입국일'] = np.where(pd.notnull(df_temp['ERCCR_DT']) == True, df_temp['ERCCR_DT'], df_temp['입국일'])
df_temp['출국일'] = np.where(pd.notnull(df_temp['LEAVE_DT']) == True, df_temp['LEAVE_DT'], df_temp['출국일'])

In [ ]:
df_temp.출발일 = df_temp.출발일.map(lambda x: x.to_pydatetime() if str(type(x)) == "<class 'pandas._libs.tslibs.timestamps.Timestamp'>" else x)

In [ ]:
df_temp['입국(예정)일'] = df_temp['입국(예정)일'].map(lambda x: x.strftime("%Y%m%d") if str(type(x)) == "<class 'datetime.datetime'>" else np.nan)
df_temp.출발일 = df_temp.출발일.map(lambda x: x.strftime("%Y%m%d") if str(type(x)) == "<class 'datetime.datetime'>" else np.nan)
df_temp.입국일 = df_temp.입국일.map(lambda x: x.strftime("%Y%m%d") if x==x else np.nan)
df_temp.출국일 = df_temp.출국일.map(lambda x: x.strftime("%Y%m%d") if x==x else np.nan)
df_temp.입력일 = df_temp.입력일.map(lambda x: x.strftime("%Y%m%d") if x==x else np.nan)

In [ ]:
temp = df_temp.sort_values(by=['ERCCR_DT'], axis=0, ascending=False)

In [ ]:
#

In [ ]:
#temp

In [ ]:
#함수 정의
def hi(x):
    if x=="K1":
        return "체류자"
    elif x=="K2":
        return "출국자"
    else:
        return np.nan

In [ ]:
temp.ERCCR_YN = temp.ERCCR_YN.map(hi)

In [ ]:
#temp

In [ ]:
# 3. 학교 중요 파일 Merge#raw로 출력
temp.to_excel("raw_" + today + ".xlsx",index=False)

# 4. 관리자용 파일 만들기

In [ ]:
#관리자용 파일
ca = ['UNIV_NM','CLASS_NO','SOJO_QUALF','BIRTH','RESDNC_TYP','MBTLNUM','PASPORT_NO']
cb = ['USER_NAME','ERCCR_DT','ORGIN_COUNTRY','FRGNR_NO','LEAVE_DT','ERCCR_YN']
cc = ['코드','학번','한글성명','학생구분','現전형유형','국적','국적구분','성별','캠퍼스','통합학부','학위','학과전공']
cd = ['재학여부','학적상태','최종학적변동','최종학적변동일','설문조사응답이메일','설문조사응답전화번호']
ce = ['설문조사응답위챗','기숙사','출발일','입국일','출국일','현상태','GLS입력여부']
cf = ['GLS외부이메일','GLS내부이메일','GLS등록주소','GLS등록연락처','입국경과일','최근14일이내']
cg = ['자가격리','유증상여부','주거형태','연락처(핸드폰)(학생입력)','이메일(학생입력)','위챗(학생입력)']
ch = ['카카오톡(학생입력)','우편번호','한국거주지주소1(학생입력)','한국거주지주소2(학생입력)']

column_name = ca + cb + cc + cd + ce + cf + cg + ch

In [ ]:
temp2 = temp[temp['학번'] == temp['학번']]

In [ ]:
admin = temp2.loc[:, column_name]

admin['입국경과일2'] = admin.ERCCR_DT.map(lambda x:(math.floor((daynow - x) / np.timedelta64(1, 'D'))+1) if x==x else np.nan)
admin['14일이내'] = admin.입국경과일2.map(lambda x: "Y" if x<=14 else "N")

admin = admin.sort_values(by=['ERCCR_DT'], axis=0, ascending=False)
admin = admin.reset_index(drop=True)


admin.ERCCR_DT = admin.ERCCR_DT.map(lambda x: dt.datetime.strftime(x,'%Y%m%d')if x==x else np.nan)



admin.to_excel("관리자용_" + today + ".xlsx",index=False)

# 5. 입국 파일 만들기

In [ ]:
#병합 함수 정의
def join_students_inner(df_gov, df_skku, find_by_name=True):
    df_gov.loc[:,'코드'] = df_gov.BIRTH.map(lambda x: str(x)[-6:]) + df_gov.USER_NAME.map(lambda x: x if x==x else '')
    #df_gov['코드'] = df_world.BIRTH.map(lambda x: str(x)[-6:]) + df_world.USER_NAME.map(lambda x: x if x==x else '')
    df_gov.loc[:,'코드'] = df_gov.loc[:,'코드'].map(lambda x: x.replace(" ",""))
    df_gov.loc[:,'코드'] = df_gov.loc[:,'코드'].map(lambda x: x.replace("-",""))
    df_result = df_gov.merge(df_skku, how='left', on='코드')
    df_mid = df_gov.merge(df_skku, how='right', on='코드')
    
    #코드로 매칭된 친구와 아닌 친구를 나눈다-교육부
    condition = df_result['번호'] == df_result['번호']
    df_temp = df_result[condition]
    #df_temp.to_excel("임시1_" + today + ".xlsx",index=False)
    df_temp2 = df_result[~condition]
    df_temp2 = df_temp2.loc[:, df_temp2.columns[:25]]
    
    #코드로 매칭된 친구와 아닌 친구를 나눈다-gls
    condition = df_mid['UNIV_NM'] == df_mid['UNIV_NM']
    df_temp3 = df_mid[condition]
    #df_temp3.to_excel("임시3_" + today + ".xlsx",index=False)
    df_temp4 = df_mid[~condition]
    df_temp4 = df_temp4.loc[:, df_temp4.columns[25:]]
    
    df_result2 = df_temp2.merge(df_temp4, how='inner', left_on ='CLASS_NO', right_on = '학번')
    #df_result2.to_excel("임시2_" + today + ".xlsx",index=False)
    df_result3 = pd.concat([df_temp,df_result2])
    #df_result2 = df_result.merge(df_skku, how='outer', left_on ='CLASS_NO', right_on = '학번')
    return df_result3

In [ ]:
#inner join
#df_world.loc[:,'코드'] = df_world.BIRTH.map(lambda x: str(x)[-6:]) + df_world.USER_NAME.map(lambda x: x if x==x else '')
#df_world.loc[:,'코드'] = df_world.loc[:,'코드'].map(lambda x: x.replace("-",""))
#df_world.loc[:,'코드'] = df_world.loc[:,'코드'].map(lambda x: x.replace(" ",""))
#dtemp = df_world.merge(df_skku, how='inner', on='코드')
temp = join_students_inner(df_world, df_skku)
dtemp = temp.copy()

In [ ]:
c1 = ['ERCCR_DT', 'LEAVE_DT', '코드', '학번', '한글성명', '영문성명', '국적','학생구분','캠퍼스', '통합학부', '학위', '학과전공']
c2 = ['기숙사','설문조사응답이메일', 'GLS외부이메일', 'GLS내부이메일','MBTLNUM']
column_name = c1 + c2

In [ ]:
#df_in = dtemp.ERCCR_DT
#2020.4.10부터 입국한 사람의 데이터만 출력
df_in = dtemp[dtemp.ERCCR_DT >= dt.datetime(year=2020, month=4, day=10)]
df_in = df_in.sort_values(by=['ERCCR_DT'], axis=0, ascending=False)
df_in = df_in.loc[:, column_name]
df_in.rename(columns = {"ERCCR_DT": "입국일"}, inplace = True)
df_in.rename(columns = {"LEAVE_DT": "출국일"}, inplace = True)
#날짜 형식 바꾸기
df_in.입국일 = df_in.입국일.map(lambda x: dt.datetime.strftime(x,'%Y%m%d')if x==x else np.nan)
df_in.출국일 = df_in.출국일.map(lambda x: dt.datetime.strftime(x,'%Y%m%d') if x==x else np.nan)
df_in.to_excel("입국_" + today + ".xlsx",index=False)

# 5-1. 입국자시스템업데이트요청 만들기

In [ ]:
df_update = temp.copy()
column_name = ['학번','ERCCR_DT','MBTLNUM']
df_update = df_update.loc[:, column_name]
df_update = df_update.sort_values(by=['ERCCR_DT'], axis=0, ascending=False)
df_update.ERCCR_DT = df_update.ERCCR_DT.map(lambda x: dt.datetime.strftime(x,'%Y%m%d')if x==x else np.nan)
#df_update.ERCCR_DT = df_update.ERCCR_DT.map(lambda x: dt.datetime.strptime(x,'%Y%m%d'))

df_update.rename(columns = {'ERCCR_DT': "입국일"}, inplace = True)
df_update.rename(columns = {'MBTLNUM': "교육부제공연락처"}, inplace = True)

df_update['입국예정구분'] = '기입국'
df_update['현상태'] = '입국'

df_update.to_excel("입국자시스템업데이트요청_" + today + ".xlsx",index=False)

# 6. 출국 파일 만들기

In [ ]:
#2020.4.10부터 출국한 사람의 데이터만 출력
c1 = ['ERCCR_DT', 'LEAVE_DT', '코드', '학번', '한글성명', '영문성명', '국적','학생구분','캠퍼스', '통합학부', '학위', '학과전공']
c2 = ['기숙사','설문조사응답이메일', 'GLS외부이메일', 'GLS내부이메일','MBTLNUM']
column_name = c1 + c2

dtemp = temp.copy()

df_out = dtemp[dtemp.LEAVE_DT >= dt.datetime(year=2020, month=4, day=10)]
df_out = df_out.sort_values(by=['LEAVE_DT'], axis=0, ascending=False)
df_out = df_out.loc[:, column_name]
df_out.rename(columns = {"ERCCR_DT": "입국일"}, inplace = True)
df_out.rename(columns = {"LEAVE_DT": "출국일"}, inplace = True)

In [ ]:
#날짜 형식 바꾸기
df_out.입국일 = df_out.입국일.map(lambda x: dt.datetime.strftime(x,'%Y%m%d')if x==x else np.nan)
df_out.출국일 = df_out.출국일.map(lambda x: dt.datetime.strftime(x,'%Y%m%d') if x==x else np.nan)
df_out.to_excel("출국_" + today + ".xlsx",index=False)# 3. 학교 중요 파일 Merge

# 7. 증상체크 파일 만들기

In [ ]:
df_world_join.입국일

In [ ]:
df_check1 = df_world_join[(df_world_join.입국일 >= daynow - datetime.timedelta(days=14)) | (df_world_join.ERCCR_DT >= daynow - datetime.timedelta(days=14))]

In [ ]:
#df_check1

In [ ]:
print(df_check1.shape)

In [ ]:
df_check1.ERCCR_DT.replace(np.nan,df_check1.입국일,inplace=True)
df_check1 = df_check1.sort_values(by=['ERCCR_DT'], axis=0, ascending=False)
df_check1 = df_check1.reset_index(drop=True)

In [ ]:
#df_check1.loc[:,df_check1.columns[106:120]]

In [ ]:

#df_check1.체류경과일
#df_check1['staydays'] = df_check1.입국일.map(lambda x:(math.floor((daynow - x) / np.timedelta64(1, 'D'))+1) if x==x else np.nan)
df_check1['staydays'] = df_check1.ERCCR_DT.map(lambda x:(math.floor((daynow - x) / np.timedelta64(1, 'D'))+1) if x==x else np.nan)

#증상여부체크
df_check1['증상여부checknum'] = df_check1.loc[:,df_check1.columns[92:106]].sum(axis = 1)
df_check1['증상여부checknum'] = df_check1.증상여부checknum.map(lambda x: math.floor(x))
df_check1['증상여부체크'] = df_check1['증상여부checknum'] > 0

#자가격리체크
df_check1['자가격리checknum'] = df_check1.loc[:,df_check1.columns[106:120]].sum(axis = 1)
#df_check1['checknum'] = df_check1.loc[:,df_check1.columns[89:117]].sum(axis = 1)
df_check1['자가격리checknum'] = df_check1.자가격리checknum.map(lambda x: math.floor(x))
df_check1['자가격리체크'] = df_check1['자가격리checknum'] == df_check1['staydays']
#df_check1[df_check1['체크'] == True]
#증상 체크를 안한 사람만 출력
#df_check = df_check1[df_check1['체크'] == False]
df_check = df_check1

In [ ]:
df_check1

In [ ]:
#컬럼 정리
c1 = ['USER_NAME','SOJO_QUALF','MBTLNUM','ERCCR_DT','입국일','ERCCR_YN','코드','학번','한글성명']
c2 = ['영문성명','국적','성별','학생구분','학위','학과전공','학적상태','GLS외부이메일','GLS내부이메일']
c3 = ['등록자','등록일','수정자','수정일','staydays','자가격리checknum','자가격리체크','증상여부checknum','증상여부체크']
column_name = c1 + c2 + c3
df_check = df_check.loc[:, column_name]
df_check.rename(columns = {"MBTLNUM": "전화번호"}, inplace = True)
#df_check.rename(columns = {"ERCCR_DT": "입국일"}, inplace = True)
#df_check.rename(columns = {"LEAVE_DT": "출국일"}, inplace = True)
df_check.rename(columns = {"ERCCR_YN": "체류여부"}, inplace = True)
#날짜 형식 바꾸기
df_check.입국일 = df_check.입국일.map(lambda x: dt.datetime.strftime(x,'%Y%m%d')if x==x else np.nan)
df_check.ERCCR_DT = df_check.ERCCR_DT.map(lambda x: dt.datetime.strftime(x,'%Y%m%d')if x==x else np.nan)
#df_check.출국일 = df_check.출국일.map(lambda x: dt.datetime.strftime(x,'%Y%m%d') if x==x else np.nan)

In [ ]:
#증상 체크를 안한 사람만 출력
df_check.to_excel("증상체크_" + today + ".xlsx",index=False)